## Learning-to-Rank (LTR) Model Training with LightGBM

In the previous notebook, we generated LTR feature datasets containing user_id, item_id, label (implicit positive signals), group_id (user-level grouping), ALS user & item embeddings (dense features). This notebook trains a **Listwise LightGBM LTR model** using the features produced earlier.

### Agenda

1. We will first load LTR datasets  
2. Then, we will prepare LightGBM ranking datasets  
3. After this, we will train a **Listwise LightGBM model**  
4. Next, we will evaluate ranking quality using **NDCG@10**  
5. Finally, we wil export model + predictions + metrics

### Key Takeaways

From this exercise, we will learn:
- How to train a *listwise* ranking model using LightGBM’s ranking mode  
- How user-level grouping controls the ranking objective  
- How to compute NDCG@10 for validation and test splits  
- How LTR models convert rich features into improved ranking performance

### Setup

In [1]:
! pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from tqdm import tqdm
import json
from pathlib import Path

In [3]:
import json
from pathlib import Path

base_path = Path(
    "/content/drive/MyDrive/upgrad_live_sessions/"
    "Recommendation_systems/notebook-1/C6/data_splits/"
)

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Verify the files are visible
import os
files = os.listdir(base_path)
print(files)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# import json
# from pathlib import Path

# base_path = Path(
 #    "/content/drive/MyDrive/upgrad_live_sessions/"
#     "Recommendation_systems/notebook-1/C6/data_splits/"
 # )

### Load LTR Feature Datasets

In [ ]:
# base_path = "/content/"
train = pd.read_parquet(base_path / "ltr_train.parquet")
valid = pd.read_parquet(base_path / "ltr_valid.parquet")
test  = pd.read_parquet(base_path / "ltr_test.parquet")

train.head()

### Identify Feature Columns

Next, let us decode the feature columns and specify:
- Labels: `label`
- Query groups: `group_id` (each user = 1 group)
- Features: all vector columns

In [ ]:
feature_cols = [c for c in train.columns if c.startswith("user_vec_") or c.startswith("item_vec_")]

len(feature_cols), feature_cols[:5]

### Build LightGBM Ranking Datasets

In [ ]:
def build_lgb_dataset(df):
    X = df[feature_cols].astype("float32")
    y = df["label"].astype("float32")

    # group sizes = number of rows per user
    group = df.groupby("group_id").size().tolist()

    return lgb.Dataset(X, label=y, group=group)

train_ds = build_lgb_dataset(train)
valid_ds = build_lgb_dataset(valid)

In [ ]:
display(valid["label"].value_counts())
display(train["label"].value_counts())

### Train LightGBM Listwise Ranking Model

Now, we will train the LightGBM listwise ranking model.
We will use:
- **lambdarank** objective (listwise ranking)
- Metric = **NDCG@10**

In [ ]:
params = {
    "objective": "lambdarank",
    "metric": "ndcg@10",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 40,
    "feature_pre_filter": False,
    "verbosity": -1,
}

In [ ]:
model = lgb.train(
    params,
    train_set=train_ds,
    valid_sets=[valid_ds],
    num_boost_round=500,
)

In [ ]:
model.save_model("lgb_ltr_model.txt")
print("Model trained & saved.")

### Predict Scores for Validation & Test Splits


In [ ]:
def rank_split(df):
    X = df[feature_cols].astype("float32")
    df["score"] = model.predict(X, num_iteration=model.best_iteration)
    return df

In [ ]:
valid_scored = rank_split(valid.copy())
test_scored  = rank_split(test.copy())

valid_scored.head()

In [ ]:
test_scored.head()

### Compute NDCG@10

We compute NDCG per-user, then average across users.

In [ ]:
def ndcg_at_k(labels, scores, k=10):
    """Compute NDCG@k for a single user."""
    df = pd.DataFrame({"label": labels, "score": scores})
    df = df.sort_values("score", ascending=False).head(k)

    dcg = (df["label"] / np.log2(np.arange(2, len(df)+2))).sum()
    ideal = (df["label"].sort_values(ascending=False) / np.log2(np.arange(2, len(df)+2))).sum()
    return dcg / ideal if ideal > 0 else 0.0

In [ ]:
def compute_ndcg(df, k=10):
    scores = []
    for gid, group in df.groupby("group_id"):
        scores.append(ndcg_at_k(group.label.values, group.score.values, k=k))
    return float(np.mean(scores))

In [ ]:
ndcg_val  = compute_ndcg(valid_scored, k=10)
ndcg_test = compute_ndcg(test_scored,  k=10)

In [ ]:
ndcg_val, ndcg_test

### Saving the Predictions

In [ ]:
valid_scored.to_parquet("ltr_predictions_val.parquet")
test_scored.to_parquet("ltr_predictions_test.parquet")
print("Predictions saved.")

### Saving the metrics

In [ ]:
metrics = {
    "ndcg@10_val": ndcg_val,
    "ndcg@10_test": ndcg_test,
}

with open("ltr_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

metrics

### Conclusion

In this exercise, we trained a **Listwise LightGBM LTR model** on the
user–item interaction features generated in previous notebook. We computed **NDCG@10** on the validation and test splits,demonstrating how ranking models use group-level objectives to optimise the quality of top-N recommendations.